# ArNet2 - Preprocessing and Model Prediction

This notebook demonstrates the workflow for:
1. **Preprocessing R-peaks annotation data** from the **SHDB-AF** dataset for prediction.
2. **Running the trained ArNet2 model** for **AF prediction** using the preprocessed data.

---

### Dataset:
- **SHDB-AF**: The dataset used in this example is the [SHDB-AF dataset](https://physionet.org/content/shdb-af/1.0.1/), which contains ECG signals annotated with peak information and AF-related labels.

### Dataset Details:
- **ECG signals**: The data consists of ECG recordings, where each sample is labeled with corresponding **R-peaks annotations**.
- **R-peak annotations**: The location of R-peaks in the ECG signal.
- **AF labels per peak**: Each R-peak is annotated with a label indicating whether it is associated with **AF** or not.
- **Overall patient label**: The dataset includes a **global label** for each patient indicating the overall AF status (e.g., **PAF**: Paroxysmal AF, **Per**: Persistent AF, **Non-AF**).

This notebook will help demonstrate how to prepare the data for prediction and how to use the trained **ArNet2 model** to make predictions for **AF detection**.


### 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import wfdb
import requests
import pickle
import subprocess
from tqdm.notebook import tqdm  # Import tqdm for Jupyter Notebooks

### 2. Setting Up the Paths
We will first set up paths for data storage and where to save the processed data:

In [2]:
# Setting up the paths
data_path = '.././physionet_data'  # Where you'll download the PhysioNet dataset
output_path = '.././data'  # Where to save the processed data
os.makedirs(data_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

window_size=60

### 3. Download the Data
Next, we will download the required ECG signal data and annotation files for SHDB-AF:

In [3]:
def download_file(url, filename):
    """
    Downloads a file from a specified URL and saves it to the given filename.

    :param url: URL to the file to be downloaded.
    :param filename: Path where the downloaded file will be saved.
    """
    response = requests.get(url)
    with open(filename, 'wb') as f:
        f.write(response.content)


def url_exists(url):
    """Return True if the URL exists (status code 200)."""
    try:
        response = requests.head(url, allow_redirects=True, timeout=10)
        return response.status_code == 200
    except requests.RequestException:
        return False

In [4]:
# Download annotation files and additional data
download_file(f'https://physionet.org/files/shdb-af/1.0.1/AdditionalData.csv', f'{data_path}/AdditionalData.csv')

additionaldata = pd.read_csv(f'{data_path}/AdditionalData.csv')
record_names = additionaldata.Data_ID.astype(str).str.zfill(3)

In [7]:
for record_name in tqdm(record_names, desc="Downloading ECG records", leave=False):
    filepath = f'{data_path}/{record_name}'
    url = f'https://physionet.org/files/shdb-af/1.0.1/{record_name}'
    #  download records and annotations
    if url_exists(f"{url}.atr") and not os.path.exists(f"{filepath}.atr"):
        download_file(f'{url}.atr', f'{filepath}.atr')
        download_file(f'{url}.qrs', f'{filepath}.qrs')

### 4. Preprocess annotation Data

We will preprocess the R-peaks, time stamps and associated recording ids

In [8]:
def pad_rhythm(rhythm, missing=None):
    """
    Helper function which receives the changes in the cardiac rhythm labels and pads the whole vector.
        Example:
            in = ['AFIB', '', '', '', '', '', 'N', '', '', '', 'SBR', '']
            out = ['AFIB', 'AFIB', 'AFIB', 'AFIB', 'AFIB', 'AFIB', 'N', 'N', 'N', 'N', 'SBR', 'SBR']
        This function in used to parse the '.bea' files summarizing the beats detected in the UVAF database.
    :param rhythm: The input vector representing the changes in the cardiac rhythm (list of strings or labels).
    :param missing: The different strings or labels (list) which characterize a missing rhythm. (If None, considering only '' as a missing rhythm)
    :returns rhythm: The padded vector of rhythms.
    """
    cond = np.ones(len(rhythm), dtype=bool)
    if missing != None:
        for char in missing:
            cond = np.logical_and(cond, rhythm != char)
    else:
        cond = rhythm != missing
    not_none = np.where(cond)[0]
    if len(not_none) == 0:
        rhythm = np.array(['(N'] * len(rhythm))
    else:
        not_none = np.append(not_none, len(rhythm))
        diffs = np.diff(not_none)
        sing_rhy = rhythm[not_none[:-1]]
        rhythm[not_none[0]:] = np.repeat(sing_rhy, diffs)
        rhythm[0:not_none[0]] = sing_rhy[0]
    return rhythm


def calc_y(rhythm, window_size):
    """
    Generates window labels based on the rhythm sequence.

    :param rhythm: The sequence of rhythm labels (AF or non-AF).
    :param window_size: The size of each window in beats.
    :returns: Binary array with labels (1 for AF, 0 for non-AF).
    """
    rhythms = pad_rhythm(np.array(rhythm), missing=['', 'None'])
    rlab = rhythms[:(len(rhythms) // window_size) * window_size].reshape(-1, window_size)
    counts = np.sum((rlab == '(AFIB'), axis=1)
    return counts >= window_size // 2  # If half or more are AF, label as AF


In [9]:
all_dfs = []  # list to collect per-record features DataFrames
all_y_df = []  # list to collect per-record true labels Dataframes
for record_name in tqdm(record_names, desc="Processing ECG records", leave=False):
    filepath = f"{data_path}/{record_name}"

    try:
        if not os.path.exists(f"{filepath}.atr"):
            continue
        # Load annotations (e.g., R-peaks)

        else:
            annotation = wfdb.rdann(filepath, 'atr')

            # Extract R-peak sample indices
            r_peaks = annotation.sample

            # Skip files with too few beats
            if len(r_peaks) < 2:
                print(f"Skipping {record_name}: not enough peaks ({len(r_peaks)})")
                continue

            # Calculate RR intervals and convert to seconds
            rr_intervals = np.diff(r_peaks).astype(np.float32)
            rr_time = r_peaks[1:] / annotation.fs
            rr_data = rr_intervals / annotation.fs

            #  derive y labels
            y = calc_y(annotation.aux_note, window_size)

            # Create per-record DataFrame
            ecg_df = pd.DataFrame({
                'rr_data': rr_data,
                'rr_time': rr_time,
                'rec_id': record_name,
            })

            y_df = pd.DataFrame({
                'true_label': y,
                'prec_window': np.arange(y.size, dtype=int),
                'rec_id': record_name,
            })

            # Append to the list
            all_dfs.append(ecg_df)
            all_y_df.append(y_df)
    except Exception as e:
        print(f"Error processing {record_name}: {e}")
        continue

# Concatenate all DataFrames into one
combined_df = pd.concat(all_dfs, ignore_index=True)
combined_y_df = pd.concat(all_y_df, ignore_index=True)


Processing ECG records:   0%|          | 0/128 [00:00<?, ?it/s]

### 5. Save the Preprocessed Data for Prediction
We save the processed R-peak annotation data as a CSV file, which will be used for prediction:

In [10]:
# Save the DataFrame as CSV
predict_file_path = os.path.join(output_path, 'shdb_df_test.csv')
combined_df.to_csv(predict_file_path, index=False)

print("ECG data processing complete. File saved as shdb_df_test.csv")

ECG data processing complete. File saved as shdb_df_test.csv


### 6. Run Prediction with the Trained Model
Finally, we use the trained ArNet2 model to make predictions based on the preprocessed data:

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
config_file_path = os.path.join(project_root, 'config', 'config.yml')
import os
import yaml

# Assuming config.yml is in the ./config/ directory of your project
config_file_path = os.path.join(project_root, 'config', 'config.yml')

# Load the config file (YAML format assumed)
with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

# Convert relative paths to absolute paths based on project_root
config['path']['arnet2'] = os.path.abspath(os.path.join(project_root, config['path']['arnet2']))
config['path']['resnet'] = os.path.abspath(os.path.join(project_root, config['path']['resnet']))

# Save the updated config back to the file
updated_config_file_path = os.path.join(project_root, 'config', 'config_w_abs_path.yml')

with open(updated_config_file_path, 'w') as file:
    yaml.dump(config, file, default_flow_style=False)

subprocess.run(['python', '../run_ArNet2.py', '--mode', 'predict', '--input_file', predict_file_path, '--output_name', 'shdb_pred_df', '--save_output_path', '../results/', '--config', updated_config_file_path])

### 8. Run Model Evaluation


This section evaluates the performance of the **ArNet2** model on the prediction results using various performance metrics such as **accuracy**, **F1-score**, **sensitivity**, **specificity**, **AUROC**, and **AUPRC**.

We will compare the model's predicted labels and the true labels to calculate these metrics.

In [12]:
import utils.metrics as metrics
pred_df = pd.read_csv('../results/shdb_pred_df.csv')
pred_df.rec_id = pred_df.rec_id.astype(str).str.zfill(3)

# Merge true label per window to prediction df
combined_pred_df = pred_df.merge(combined_y_df[['rec_id', 'prec_window', 'true_label']], on=['rec_id', 'prec_window'], how='left')

EmptyDataError: No columns to parse from file

In [15]:
import utils.metrics as metrics
pred_df = pd.read_csv('./../results/shdb_pred_df.csv')
# pred_df.rec_id = pred_df.rec_id.astype(str).str.zfill(3)


EmptyDataError: No columns to parse from file

In [26]:
accuracy, fbeta, sensitivity, specificity, PPV, NPV, AUROC, AUCPR = metrics.model_metrics(combined_pred_df.proba, combined_pred_df.true_label, pred_df.pred, print_metrics=True)

Accuracy: 0.7451340433345575
F1-Score: 0.7573426573426573
Sensitivity: 1.0
Specificity: 0.5768292682926829
PPV: 0.6094541361845808
NPV: 1.0
AUROC: 0.8196386646172558
AUCPR: 0.7487616942816266
[[1892 1388]
 [   0 2166]]
